# D3 projected NYC points

Project the original longitude/latitude coordinates with D3's Mercator projection. Each point is one synthetic observation; its radius encodes the selected count or total metric. This is a projected point display **without a basemap**.

Run every cell top to bottom. Python uses only the standard library for preparation; `IPython.display.IFrame` embeds the live PyScript app. The notebook kernel and iframe each run their **own Python interpreter** and load their own dataset/module. URL parameters configure the app; changing notebook variables or data is **not shared memory**.

## 1. Inspect the shared data and provenance

These 24 synthetic NYC observations have invented hours, values and location offsets. They are not measured taxi trips, health records or observations of people.

In [ ]:
import json
from pathlib import Path
from urllib.parse import urlencode
from IPython.display import IFrame, display
from chart_data import chart_config, chart_summary, export_summary

dataset = json.loads(Path('data/nyc-observations.geojson').read_text())
print(dataset['metadata']['provenance'])
print('Synthetic observations:', len(dataset['features']))
assert dataset['metadata']['simulated'] is True

## 2. Set parameters and prepare real Python rows

Use `hour = 'all'` for all 24 observations, `8` for six observations, or `0` for a valid empty selection. Choose `metric = 'total'` for the additive invented value, or `'count'` for the number of observations. Re-run from this cell after changing parameters.

In [ ]:
hour = 'all'
metric = 'total'
renderer = 'd3'
view = 'points'
group_by = 'point'
config = chart_config(renderer, view, hour, metric)
summary = chart_summary(dataset, group_by, config['hour'], config['metric'])
print('Selected observations:', summary['metadata']['feature_count'])
print('Summary groups:', len(summary['rows']))
for row in summary['rows']:
    print(row['id'], row['label'], 'count =', row['count'], 'total =', row['total'], 'plotted =', row['value'])

## 3. Verify the summary and empty selection

Counts and totals are additive. Neighborhood coordinates are arithmetic centroids of the selected synthetic points; individual point rows retain their original coordinates. The export retains dataset provenance.

In [ ]:
assert sum(row['count'] for row in summary['rows']) == summary['metadata']['feature_count']
assert all(row['value'] == row[metric] for row in summary['rows'])
assert json.loads(export_summary(dataset, group_by, hour, metric)) == summary
empty = chart_summary(dataset, group_by, 0, metric)
assert empty['rows'] == []
assert empty['metadata']['feature_count'] == 0
print('Python preparation and empty-selection checks passed.')

## 4. Run the interactive PyScript app here

The build writes `hub-config.json` with the actual deployment prefix. This produces a root-relative iframe URL that bypasses JupyterLab's notebook-file resolver and works at a site root or repository path. The app loads its own copy of `chart_data.py` and the dataset. Controls inside the iframe update that app; they do not change `summary` above.

[Launch this view standalone]({{HUB_BASE}}/apps/d3/?view=points&hour=all&metric=total) · [Chart chapter]({{HUB_BASE}}/book/d3-and-d3plus/).

In [ ]:
hub = json.loads(Path('hub-config.json').read_text())
query = urlencode({'view': view, 'hour': config['hour'], 'metric': config['metric']})
app_url = f"{hub['base_url']}/apps/{renderer}/?{query}"
print('Embedded app:', app_url)
display(IFrame(src=app_url, width='100%', height=960, extras=['title="Interactive D3 projected NYC points"']))

## 5. Explore the deployed source

The build packages the actual app and shared sources below. JavaScript draws the serialized Python rows; it does not perform filtering or aggregation. Changing these notebook copies does not change the deployed app.

In [ ]:
print(Path('chart_data.py').read_text())

In [ ]:
print(Path('source/charts.py').read_text())
print(Path('source/apps/d3/index.html').read_text())

Read [the actual renderer]({{HUB_BASE}}/apps/shared/charts.js) and [runtime file mapping]({{HUB_BASE}}/apps/shared/charts.json). In JupyterLite, open `source/charts.js` to inspect the complete drawing code.

## Exercises

Inspect the fixed projection bounds and `geoMercator().fitExtent(...)`. Explain why geographic coordinates remain unchanged in the Python rows while SVG coordinates change when the viewport resizes.

1. Set `hour = 8` and predict the group count before rerunning. Compare the table and downloaded JSON inside the iframe with `summary`.
2. Change `metric` to `'count'`, rerun the parameter and iframe cells, then compare the mark sizes.
3. Set `hour = 0` and verify every data mark disappears, the table explains the empty selection and exported `rows` is empty.
4. Change a value in the notebook's `dataset` and rerun only its summary: explain why the embedded app keeps its own data. Restore the dataset by rerunning cell 1.

Download important work from the JupyterLite file browser before clearing browser storage.